<a href="https://colab.research.google.com/github/hsandmann/biblio/blob/main/material/handouts/regressao-do-zero.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Regressão linear, do zero

**Machine Learning · Insper**

Este roteiro percorre um caminho só, sem atalho:

1. escrever o modelo em forma matricial, $\hat{y} = Xw$;
2. tentar a saída óbvia — inverter $X$ — e ver, rodando, por que ela não existe;
3. chegar às equações normais, resolvendo uma delas no papel;
4. medir onde essa solução quebra quando o número de atributos cresce;
5. de bônus, aplicar essa mesma conta a um ano de Ibovespa e tentar prever onde o
   índice fecha o ano.

O gradiente descendente, que é a saída para o caso em que a forma fechada não cabe na
memória, fica para a próxima aula.

Nenhuma célula depende de biblioteca de ML. Só `numpy` para as contas de matriz e
`matplotlib` para ver — mais o `yfinance` na seção bônus, que serve apenas para baixar
os dados. Rode as células em ordem.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

np.set_printoptions(precision=4, suppress=True)
print("pronto")

## 1. Cinco pontos

Números pequenos de propósito: tudo o que vier a seguir pode ser conferido à mão.

In [ ]:
x = np.array([1., 2., 3., 4., 5.])
y = np.array([2., 4., 5., 4., 6.])

plt.figure(figsize=(5,3.2))
plt.scatter(x, y, s=45, color='#f0883e', zorder=3)
plt.xlabel('x'); plt.ylabel('y'); plt.grid(alpha=.3)
plt.title('cinco observações'); plt.show()

print("x =", x)
print("y =", y)

## 2. A forma matricial

O modelo é $\hat{y}_i = w_0 + w_1 x_i$. Empilhando as observações e **absorvendo o
intercepto como uma coluna de 1s**, isso vira uma única multiplicação de matrizes:

$$
\hat{y} = Xw,
\qquad
X = \begin{pmatrix} 1 & x_1 \\ \vdots & \vdots \\ 1 & x_n \end{pmatrix},
\qquad
w = \begin{pmatrix} w_0 \\ w_1 \end{pmatrix}
$$

A coluna de 1s não é truque de notação: é o que faz o intercepto ser tratado como
qualquer outro peso.

In [ ]:
X = np.column_stack([np.ones_like(x), x])
print("X =")
print(X)
print("\nforma de X:", X.shape, " ->", X.shape[0], "observações,", X.shape[1], "parâmetros")

## 3. A tentativa óbvia: $w = X^{-1}y$

Se $\hat{y} = Xw$, o reflexo é isolar $w$ invertendo $X$. Rode e veja o que acontece.

In [ ]:
try:
    w = np.linalg.inv(X) @ y
    print("w =", w)
except np.linalg.LinAlgError as erro:
    print("LinAlgError:", erro)

### Primeiro obstáculo: $X$ não é quadrada

`inv` só aceita matriz quadrada e $X$ é $5 \times 2$. No caso geral ela é $n \times d$ com $n \gg d$ — milhares de linhas, poucas colunas.

### Segundo obstáculo, o que realmente importa

Mesmo ignorando o formato: **não existe solução exata**. O sistema $y = Xw$ tem 5 equações e 2 incógnitas — é *sobredeterminado*. Um $w$ exato só existiria se $y$ estivesse no espaço gerado pelas colunas de $X$.

O posto responde isso. Se acrescentar $y$ a $X$ **aumenta** o posto, é porque $y$ aponta para uma direção que as colunas de $X$ não alcançam.

In [ ]:
posto_X   = np.linalg.matrix_rank(X)
posto_aug = np.linalg.matrix_rank(np.column_stack([X, y]))

print("posto(X)      =", posto_X)
print("posto([X | y]) =", posto_aug)
print()
if posto_aug > posto_X:
    print("y NÃO está no espaço-coluna de X.")
    print("=> nenhum w satisfaz Xw = y exatamente.")
else:
    print("y está no espaço-coluna: existe solução exata.")

Isso não é azar deste conjunto: é a situação normal. Se existisse $w$ exato, todos os resíduos seriam zero e a reta passaria por todos os pontos. Com dados ruidosos isso não acontece — **e é justamente por isso que a regressão existe**.

## 4. O que fazemos no lugar

Como $y$ está fora de alcance, buscamos o ponto alcançável **mais próximo** dele: minimizar $\lVert y - Xw \rVert^2$. Igualando o gradiente a zero chega-se às **equações normais**:

$$X^\top X\, w = X^\top y$$

O truque está no formato: $X^\top X$ é $d \times d$ — **quadrada** — e invertível sempre que $X$ tem posto-coluna cheio. Nunca invertemos $X$; invertemos $X^\top X$.

In [ ]:
XtX = X.T @ X
Xty = X.T @ y

print("XᵀX =");  print(XtX);  print("forma:", XtX.shape, "-> quadrada")
print("\nXᵀy =", Xty)

w = np.linalg.solve(XtX, Xty)     # resolve o sistema; NÃO usa inv()
print("\nw =", w)
print(f"reta ajustada:  ŷ = {w[0]:.1f} + {w[1]:.1f}·x")

### Conferindo no papel

Com estes cinco pontos o sistema é pequeno o bastante para resolver à mão:

$$
X^\top X = \begin{pmatrix} 5 & 15 \\ 15 & 55 \end{pmatrix},
\qquad
X^\top y = \begin{pmatrix} 21 \\ 71 \end{pmatrix}
$$

$$
\begin{cases} 5w_0 + 15w_1 = 21 \\ 15w_0 + 55w_1 = 71 \end{cases}
$$

- Da primeira,

    $$\displaystyle w_0 = \frac{21 - 15w_1}{5}$$.

- Substituindo na segunda:

    \begin{align}
    \displaystyle 15 \left( \frac{21 - 15w_1}{5} \right) + 55w_1 & = 71 \\
    \displaystyle 3 \left( 21 - 15w_1 \right) + 55w_1 & = 71 \\
    63 - 45w_1 + 55w_1 & = 71 \\
    10w_1 & = 8 \\
    w_1 & = 0{,}8
    \end{align}
    
    portanto,

    \begin{align}
    \displaystyle w_0 &= \frac{21 - 15w_1}{5} \\
    &= 1{,}8
    \end{align}

    ao final:

    $\mathbf{w_1 = 0{,}8}$ e $\mathbf{w_0 = 1{,}8}$

Confira contra a saída da célula acima.

In [ ]:
yhat = X @ w
resid = y - yhat

print("ŷ =", yhat)
print("e =", resid)
print()
print("Σe      =", round(resid.sum(), 12) + 0.0)
print("Σe·x    =", round((resid * x).sum(), 12) + 0.0)
print("Xᵀe     =", (X.T @ resid).round(12) + 0.0)
print()
print("Os dois zeros acima SÃO as equações normais, escritas por extenso:")
print("uma por parâmetro. O resíduo é ortogonal a cada coluna de X.")

In [ ]:
plt.figure(figsize=(5,3.2))
plt.scatter(x, y, s=45, color='#f0883e', zorder=3, label='dados')
xs = np.linspace(0.5, 5.5, 50)
plt.plot(xs, w[0] + w[1]*xs, color='#1F6B8E', lw=2, label=f'ŷ = {w[0]:.1f} + {w[1]:.1f}x')
for xi, yi, yh in zip(x, y, yhat):
    plt.plot([xi, xi], [yi, yh], color='#9E3B2E', lw=1.2, alpha=.8)
plt.legend(); plt.grid(alpha=.3); plt.xlabel('x'); plt.ylabel('y')
plt.title('os segmentos vermelhos são os resíduos'); plt.show()

## 5. Onde a forma fechada quebra

A fórmula funciona. A pergunta é **até quando**.

Duas coisas crescem com o número de atributos $d$:

- **tempo**: resolver o sistema custa $O(d^3)$;
- **memória**: $X^\top X$ tem $d \times d$ elementos, independentemente de quantas observações você tenha.

Meça o tempo você mesmo. Se o custo é cúbico, dobrar $d$ deveria multiplicar o tempo por $2^3 = 8$. Nos tamanhos pequenos você verá menos que isso — o custo fixo de chamar a rotina e o paralelismo interno da BLAS (Basic Linear Algebra Subprograms) ainda pesam mais que a conta. A razão sobe em direção a 8 conforme $d$ cresce.

In [ ]:
rng = np.random.default_rng(0)
print(f"{'d':>6} {'tempo':>10} {'razão vs d/2':>14}")
anterior = None
for d in [250, 500, 1000, 2000]:
    A = rng.standard_normal((d, d)); A = A @ A.T + d*np.eye(d)   # simétrica positiva-definida
    b = rng.standard_normal(d)
    t = min(( (lambda t0: (np.linalg.solve(A, b), time.perf_counter()-t0)[1])(time.perf_counter())
              for _ in range(3) ))
    razao = f"{t/anterior:.1f}×" if anterior else "—"
    print(f"{d:>6} {t:>9.4f}s {razao:>14}")
    anterior = t

O tempo é ruim, mas ainda administrável. O que **impede de vez** é a memória: $X^\top X$ é $d \times d$ e isso não depende de $n$.

In [ ]:
print("memória necessária só para GUARDAR XᵀX (float64):\n")
for d in [1_000, 10_000, 50_000, 100_000]:
    gb = d*d*8 / 1e9
    veredito = "cabe" if gb < 8 else "NÃO CABE em um notebook comum"
    print(f"  d = {d:>7}  ->  {gb:8.2f} GB   {veredito}")

print("\nUm texto vetorizado com TF-IDF facilmente passa de 100 mil colunas.")
print("Nesse caso XᵀX nem chega a ser construída — muito antes de haver o que inverter.")

## 6. Bônus: uma regressão no Ibovespa

Até aqui foram cinco pontos inventados, escolhidos para caber no papel. Agora a mesma
conta, **sem trocar uma linha da álgebra**, sobre dados reais: um ano de fechamentos da
bolsa brasileira.

O combinado é o mais simples possível:

- $y$ = o fechamento do Ibovespa;
- $x$ = a posição do pregão dentro da janela — 1, 2, 3, … até o último dia.

E o clímax: estender a reta até **31 de dezembro** para ver onde ela diz que o índice
fecha o ano.

Vale avisar desde já, porque é o ponto da seção: ao final você vai ter um número, um
gráfico bonito e uma razão muito boa para não apostar nele. Chegar até lá é o exercício.

In [ ]:
# o yfinance não vem no Colab por padrão
%pip install -q yfinance

import yfinance as yf
import pandas as pd

print("yfinance", yf.__version__)

### 6.1 Baixar um ano de pregões

O `yfinance` conversa com a base pública do Yahoo Finance. O código do Ibovespa é
`^BVSP`.

Duas observações sobre o formato do que volta:

- versões recentes da biblioteca devolvem as colunas em **dois níveis** (`Close`/`^BVSP`),
  então a célula reduz isso a uma série simples de um jeito que funciona nas duas formas;
- `dropna()` tira pregões sem cotação.

In [ ]:
bruto = yf.download("^BVSP", period="1y", progress=False, auto_adjust=True)

fech = bruto["Close"]
if isinstance(fech, pd.DataFrame):      # versões recentes devolvem colunas em dois níveis
    fech = fech.iloc[:, 0]
fech = fech.dropna()

print(f"{len(fech)} pregões, de {fech.index[0]:%d/%m/%Y} a {fech.index[-1]:%d/%m/%Y}")
print(f"primeiro fechamento: {fech.iloc[0]:>12,.0f}")
print(f"último fechamento:   {fech.iloc[-1]:>12,.0f}")
print(f"variação na janela:  {100*(fech.iloc[-1]/fech.iloc[0]-1):>11.1f}%")

### 6.2 Escolher o $x$ e o $y$

O $y$ é direto: o vetor de fechamentos.

O $x$ é a decisão interessante. Não usamos a data, usamos a **posição do pregão na
janela**: o primeiro dia é 1, o segundo é 2 e assim por diante. Isso significa tratar
todo pregão como um passo de tamanho igual — a sexta-feira e a segunda-feira seguinte
ficam a uma unidade de distância, embora separadas por três dias de calendário. Para
esta pergunta é o que queremos: fins de semana e feriados não têm preço, então não são
observações.

Repare no tamanho dos números. O $y$ está na casa das centenas de milhares e o $x$ vai
de 1 a ~250. As duas colunas de $X$ vão viver em escalas muito diferentes — guarde isso,
é o assunto de padronização que volta depois.

In [ ]:
y_ibov = fech.to_numpy(dtype=float)
x_ibov = np.arange(1, len(y_ibov) + 1, dtype=float)

print("x:", x_ibov[:6], "...", x_ibov[-3:])
print("y:", np.round(y_ibov[:6]), "...", np.round(y_ibov[-3:]))
print("\nn =", len(y_ibov))

### 6.3 A conta é exatamente a mesma

Aqui está o ponto da seção. Monte $X = [\,\mathbf{1} \;\; x\,]$ e resolva

$$(X^\top X)\,w = X^\top y$$

São as **mesmas duas linhas** da seção 4. Não há nada de financeiro no código: ele não
sabe que isso é uma bolsa de valores e trata 250 fechamentos exatamente como tratou
cinco pontos inventados.

In [ ]:
X_ibov = np.column_stack([np.ones_like(x_ibov), x_ibov])

# exatamente as mesmas duas linhas da seção 4
w_ibov = np.linalg.solve(X_ibov.T @ X_ibov, X_ibov.T @ y_ibov)

print("X_ibov.shape =", X_ibov.shape)
print("w =", w_ibov)
print(f"\nreta:  fechamento ≈ {w_ibov[0]:,.0f}  +  {w_ibov[1]:,.1f} · dia")
print(f"\nou seja: a reta atribui {w_ibov[1]:,.1f} pontos de alta a cada pregão.")

### 6.4 Ver o que saiu

Dois gráficos. O primeiro põe a reta sobre os dados. O segundo mostra os resíduos
sozinhos, que é onde mora a informação que o primeiro esconde.

Olhe o gráfico dos resíduos com atenção antes de seguir. Eles não se espalham em torno
do zero como poeira: eles **passeiam**, ficando muito tempo acima e depois muito tempo
abaixo. Segure essa imagem — a próxima célula mede exatamente isso.

In [ ]:
yhat_ibov = X_ibov @ w_ibov
res_ibov  = y_ibov - yhat_ibov

plt.figure(figsize=(9, 4))
plt.plot(x_ibov, y_ibov, color='#f0883e', lw=1.2, label='fechamento do Ibovespa')
plt.plot(x_ibov, yhat_ibov, color='#1F6B8E', lw=2,
         label=f'reta: {w_ibov[0]:,.0f} + {w_ibov[1]:,.1f}·dia')
plt.xlabel('dia (posição na janela)'); plt.ylabel('pontos')
plt.title('um ano de Ibovespa e a reta de mínimos quadrados')
plt.legend(); plt.grid(alpha=.3); plt.show()

plt.figure(figsize=(9, 2.4))
plt.axhline(0, color='#1F6B8E', lw=1.5)
plt.plot(x_ibov, res_ibov, color='#9E3B2E', lw=1)
plt.xlabel('dia'); plt.ylabel('resíduo'); plt.title('os resíduos, sozinhos')
plt.grid(alpha=.3); plt.show()

### 6.5 Quanto a reta explica

Três medidas:

- o **R²**, a fração da soma total de quadrados que a reta eliminou;
- o **$s$**, o desvio típico do resíduo, em pontos do índice — o tamanho do erro comum;
- a **correlação entre resíduos vizinhos**: o resíduo de hoje contra o de ontem.

A terceira é a que importa aqui. Uma das suposições por trás de toda inferência em
regressão é que os erros sejam **independentes**. Se o resíduo de hoje prevê o de
amanhã, essa suposição não vale e as barras de erro que a teoria oferece param de valer
junto.

Veja o número que a célula imprime. Em série de preço ele vem perto de 1 — que é a
tradução numérica daquele passeio no gráfico anterior.

In [ ]:
n_ibov = len(y_ibov)
sse = res_ibov @ res_ibov
sst = ((y_ibov - y_ibov.mean())**2).sum()
r2  = 1 - sse/sst
s   = np.sqrt(sse/(n_ibov - 2))
rho = np.corrcoef(res_ibov[:-1], res_ibov[1:])[0, 1]

print(f"R²                                = {r2:.4f}")
print(f"desvio típico do resíduo (s)      = {s:,.0f} pontos")
print(f"   ... como % do índice de hoje   = {100*s/y_ibov[-1]:.1f}%")
print(f"correlação entre resíduos vizinhos = {rho:.4f}")

### 6.6 O clímax: onde o índice fecha o ano?

Agora a extrapolação. Três passos:

1. contar quantos pregões faltam entre o último dia da janela e 31 de dezembro;
2. somar isso ao $n$ para achar a posição $x$ daquele dia;
3. jogar esse $x$ na reta.

A contagem usa `bdate_range`, que conta dias úteis — ou seja, tira sábados e domingos,
mas **não** conhece os feriados da B3. O número sai um pouco por cima, o que é
suficiente para o que queremos.

A célula imprime também um **intervalo de 95%**. Ele não é a reta ± o desvio dos
resíduos: para prever uma observação nova existe uma fórmula própria, que soma três
incertezas — a do ponto novo, a da altura da reta e a da inclinação dela:

$$\text{erro padrão} = s\sqrt{1 + \frac{1}{n} + \frac{(x_0-\bar{x})^2}{\sum (x_i-\bar{x})^2}}$$

O último termo cresce conforme $x_0$ se afasta do centro dos dados: é o preço cobrado
por extrapolar. Mas repare, quando os números saírem, que a oitenta pregões de uma
janela de duzentos e cinquenta esse castigo ainda é pequeno — o intervalo abre pouco.
Quase toda a largura vem do próprio $s$.

Isso é importante para ler o resultado direito: mesmo **sem** punição nenhuma por
extrapolar, o intervalo já nasce largo.

In [ ]:
ultima = fech.index[-1]
fim_do_ano = pd.Timestamp(ultima.year, 12, 31)

# dias úteis restantes (não desconta feriado da B3, então sai um pouco por cima)
pregoes_restantes = len(pd.bdate_range(ultima + pd.Timedelta(days=1), fim_do_ano))
dia_fim = n_ibov + pregoes_restantes

previsao = w_ibov[0] + w_ibov[1]*dia_fim

# erro padrão de PREDIÇÃO: cresce conforme x0 se afasta do centro dos dados
Sxx = ((x_ibov - x_ibov.mean())**2).sum()
se_pred = s*np.sqrt(1 + 1/n_ibov + (dia_fim - x_ibov.mean())**2/Sxx)
lo, hi = previsao - 1.96*se_pred, previsao + 1.96*se_pred

print(f"último pregão da janela: {ultima:%d/%m/%Y}  (dia {n_ibov})")
print(f"pregões até 31/12:       {pregoes_restantes}  ->  dia {dia_fim:.0f}\n")
print(f"PREVISÃO PARA O FIM DO ANO:  {previsao:,.0f} pontos")
print(f"hoje:                        {y_ibov[-1]:,.0f} pontos")
print(f"implica:                     {100*(previsao/y_ibov[-1]-1):+.1f}%\n")
print(f"intervalo de 95%: [{lo:,.0f} ; {hi:,.0f}]   (largura: {hi-lo:,.0f} pontos)")
print(f"o intervalo contém o valor de hoje? {'SIM' if lo <= y_ibov[-1] <= hi else 'não'}")

In [ ]:
xs = np.linspace(1, dia_fim, 400)
se_xs = s*np.sqrt(1 + 1/n_ibov + (xs - x_ibov.mean())**2/Sxx)
reta_xs = w_ibov[0] + w_ibov[1]*xs

plt.figure(figsize=(9, 4.4))
plt.fill_between(xs, reta_xs - 1.96*se_xs, reta_xs + 1.96*se_xs,
                 color='#1F6B8E', alpha=.12, label='intervalo de 95%')
plt.plot(x_ibov, y_ibov, color='#f0883e', lw=1.2, label='fechamento observado')
plt.plot(xs, reta_xs, color='#1F6B8E', lw=2, label='reta extrapolada')
plt.axvline(n_ibov, color='#9E3B2E', ls='--', lw=1.2)
plt.scatter([dia_fim], [previsao], color='#9E3B2E', s=70, zorder=5,
            label=f'31/12: {previsao:,.0f}')
plt.xlabel('dia (posição na janela)'); plt.ylabel('pontos')
plt.title('o clímax: a reta levada até o fim do ano')
plt.legend(loc='upper left', fontsize=9); plt.grid(alpha=.3); plt.show()

Pare no número que saiu.

A reta dá uma previsão pontual e ela é fácil de levar a sério — é específica, tem
vírgula decimal e vem com um gráfico. Agora olhe a largura do intervalo de 95% e a
última linha que a célula imprimiu.

Se o intervalo contém o valor de hoje, o modelo está dizendo, com as próprias contas,
que **não sabe nem se o índice vai subir ou cair** até dezembro. A previsão pontual não
ficou errada: ela só nunca foi uma afirmação forte. O intervalo é que mostra o tamanho
real do que se sabe.

E esse intervalo ainda é **otimista**, porque foi calculado supondo erros independentes
— exatamente a suposição que a célula 6.5 mostrou quebrada.

### 6.7 O teste honesto

Tudo isso ainda é argumento teórico. Existe uma pergunta empírica, muito mais direta:

> **este procedimento, aplicado no passado, teria acertado?**

Dá para responder sem esperar dezembro. Baixe dois anos, ajuste a reta **só no primeiro
ano** e use-a para prever o último dia do segundo ano — um horizonte de um ano, que é o
mesmo tipo de salto que fizemos até 31/12. Depois compare com o que de fato aconteceu.

A diferença entre os dois números é o erro que este método comete de verdade.

In [ ]:
d2 = yf.download("^BVSP", period="2y", progress=False, auto_adjust=True)["Close"]
if isinstance(d2, pd.DataFrame):
    d2 = d2.iloc[:, 0]
d2 = d2.dropna()

v  = d2.to_numpy(dtype=float)
n2 = len(v)
corte = n2 // 2                      # treina na primeira metade (~1 ano)

x_tr = np.arange(1, corte + 1, dtype=float)
y_tr = v[:corte]
X_tr = np.column_stack([np.ones_like(x_tr), x_tr])
w_tr = np.linalg.solve(X_tr.T @ X_tr, X_tr.T @ y_tr)

previsto = w_tr[0] + w_tr[1]*n2      # um ano à frente do fim do treino
real     = v[-1]

print(f"treino: {d2.index[0]:%d/%m/%Y} a {d2.index[corte-1]:%d/%m/%Y}  ({corte} pregões)")
print(f"alvo:   {d2.index[-1]:%d/%m/%Y}  (dia {n2})\n")
print(f"a reta previa: {previsto:>12,.0f}")
print(f"aconteceu:     {real:>12,.0f}")
print(f"erro:          {previsto-real:>+12,.0f} pontos   ({100*(previsto/real-1):+.1f}%)")

### 6.8 O que a bolsa ensinou

Compare os dois números que a célula imprimiu. O erro de um ano costuma vir na casa das
dezenas de milhares de pontos — muito maior do que o $s$ da seção 6.5 sugeria e no
mesmo horizonte que a previsão para dezembro.

Não é um defeito da conta. As equações normais devolveram a melhor reta possível para
aquela janela; a álgebra está impecável. O que falha é a **pergunta**.

Três lições, que valem muito além do Ibovespa:

1. **R² alto mede ajuste ao passado, não capacidade de prever.** São coisas diferentes
   e nada obriga uma a acompanhar a outra. A única evidência que conta sobre previsão é
   a da seção 6.7: testar em dados que o modelo não viu.

2. **Extrapolar não é o mesmo que interpolar.** Dentro da janela a reta resume os dados.
   Fora dela, ela só repete uma suposição — a de que a tendência continua — que nenhum
   dado confirmou.

3. **As suposições não são burocracia.** A correlação de quase 1 entre resíduos vizinhos
   diz que preço não oscila em torno de uma tendência: ele **passeia** e cada dia parte
   de onde o anterior chegou. Num passeio assim, a inclinação que você mediu é em boa
   parte o caminho que por acaso aconteceu, não uma força que continua atuando.

Para que serve a reta, então? Para **descrever** a janela: dizer que naquele ano o índice
andou tantos pontos por pregão, em média. Isso ela responde bem. A pergunta que ela não
responde é a que todo mundo quer fazer.

E se a resposta é "prever preço exige mais do que uma reta", o próximo passo natural é
ter um modelo grande o bastante para não caber na forma fechada — que é exatamente onde
entra o gradiente descendente, na próxima aula.

## 7. Exercícios

1. **Volte à seção 3.** Remova três observações, deixando duas. Agora $X$ é quadrada.
   Calcule `np.linalg.inv(X) @ y` e confira que funciona e que o resíduo é exatamente
   zero. Por quê?

2. Ainda com $X$ quadrada, compare `inv(X) @ y` com `inv(X.T@X) @ X.T @ y`. Eles
   coincidem. Mostre algebricamente que $(X^\top X)^{-1}X^\top = X^{-1}$ quando $X$
   é invertível.

3. **Sem intercepto.** Refaça o ajuste sem a coluna de 1s. Verifique que $\sum e_i$
   deixa de ser zero, mas $\sum e_i x_i$ continua zero. Explique usando a frase
   "uma equação normal por parâmetro".

4. **O custo, medido.** Estenda a tabela da seção 5 até `d = 4000`. A razão continua
   perto de 8? Onde ela começa a se afastar e por quê?

5. **A janela muda a tendência.** Refaça a seção 6 com `period="5y"` e depois com
   `period="6mo"`. Anote o coeficiente angular em cada caso. Se "a tendência do
   Ibovespa" fosse um número real do mundo, ele não deveria depender tanto do tamanho da
   janela que você escolheu. O que isso sugere sobre o que a inclinação está medindo?

6. **Ajuste no logaritmo.** Refaça a regressão com $y = \log(\text{fechamento})$. A
   inclinação passa a ser aproximadamente a taxa de crescimento por pregão e a previsão
   vira $e^{\hat{y}}$. Por que o log é mais defensável para preço do que o nível? (Dica:
   pense no que a reta em nível supõe sobre ganhar 100 pontos partindo de 50 mil ou de
   200 mil.)

7. **Quão ruim, em média?** Repita o teste da seção 6.7 deslizando o corte: para cada
   um de vários pontos de partida, treine em um ano e preveja o ano seguinte. Em que
   fração das vezes o erro passou de 10%? Compare com a impressão que o R² da seção 6.5
   passava.